### SourceLoader（文档加载器）

RAG 的第一步是**加载数据源**。LangChain 把所有来源统一成 `Document`：

- `page_content`：正文文本；
- `metadata`：来源、页码等结构化信息（检索/引用时会用到）。

加载器都实现 `BaseLoader` 接口：

- `.load()`：一次性返回 `list[Document]`；
- `.lazy_load()`：返回生成器，逐条产出（适合大数据集）。

下面先准备一批示例文件（txt / md / csv / json / pdf），再逐个演示常用加载器。

> 说明：这些加载器来自 `langchain-community` 包（当前已进入维护期，未来建议迁移到对应的独立包）。

In [1]:
import json
import warnings
from pathlib import Path

from rich import layout

# WebBaseLoader 会读取 USER_AGENT 标识；langchain-community 已进入维护期
warnings.filterwarnings("ignore", message=".*langchain-community is being sunset.*")
import os

os.environ.setdefault("USER_AGENT", "langchain-demo/1.0")

from langchain_community.document_loaders import (
    CSVLoader,
    DirectoryLoader,
    JSONLoader,
    PyPDFLoader,
    TextLoader,
    UnstructuredFileLoader,
    UnstructuredMarkdownLoader,
    WebBaseLoader,
)
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document

DATA_DIR = Path("temp/source_loader_demo")
DATA_DIR.mkdir(parents=True, exist_ok=True)

(DATA_DIR / "hello.txt").write_text(
    "LangChain 是一个用于构建 LLM 应用的框架。\n它支持文档加载、切分、检索。",
    encoding="utf-8",
)
(DATA_DIR / "notes.md").write_text(
    "# 标题\n\n这是 Markdown 文档。\n\n- 要点一\n- 要点二\n", encoding="utf-8"
)
(DATA_DIR / "users.csv").write_text("name,role\n小明,后端\n小红,前端\n", encoding="utf-8")
(DATA_DIR / "items.json").write_text(
    json.dumps(
        [
            {"text": "Python 适合后端", "source": "a"},
            {"text": "Vue 适合前端", "source": "b"},
        ],
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


def make_pdf(path: Path, text: str) -> None:
    """生成一页含指定文字的极简 PDF（仅用于演示，无需额外依赖）。"""
    objs = [
        "<< /Type /Catalog /Pages 2 0 R >>",
        "<< /Type /Pages /Kids [3 0 R] /Count 1 >>",
        "<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Contents 4 0 R "
        "/Resources << /Font << /F1 5 0 R >> >> >>",
    ]
    stream = f"BT /F1 24 Tf 72 700 Td ({text}) Tj ET"
    objs.append(f"<< /Length {len(stream)} >>\nstream\n{stream}\nendstream")
    objs.append("<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>")

    out = "%PDF-1.4\n"
    offsets = []
    for i, obj in enumerate(objs, start=1):
        offsets.append(len(out))
        out += f"{i} 0 obj\n{obj}\nendobj\n"
    xref_pos = len(out)
    out += f"xref\n0 {len(objs) + 1}\n0000000000 65535 f \n"
    for offset in offsets:
        out += f"{offset:010d} 00000 n \n"
    out += f"trailer\n<< /Size {len(objs) + 1} /Root 1 0 R >>\nstartxref\n{xref_pos}\n%%EOF\n"
    path.write_bytes(out.encode("latin-1"))


make_pdf(DATA_DIR / "doc.pdf", "Hello RAG PDF")


def show(tag: str, docs: list[Document]) -> None:
    print(f"[{tag}] {len(docs)} 个 Document")
    for doc in docs:
        print("  page_content:", repr(doc.page_content[:70]))
        print("  metadata:", doc.metadata)


print("示例文件已生成于：", DATA_DIR.resolve())


/var/folders/rh/3ly80g_174v93gsdvnp7n_jr0000gn/T/ipykernel_6846/1339334457.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


示例文件已生成于： /Users/lijixu/PycharmProjects/langchain_demo/charpter11-RAG/temp/source_loader_demo


#### 1. TextLoader：加载纯文本文件

In [2]:
docs = TextLoader(str(DATA_DIR / "hello.txt"), encoding="utf-8").load()
show("TextLoader", docs)

[TextLoader] 1 个 Document
  page_content: 'LangChain 是一个用于构建 LLM 应用的框架。\n它支持文档加载、切分、检索。'
  metadata: {'source': 'temp/source_loader_demo/hello.txt'}


#### 2. DirectoryLoader：批量加载整个目录

In [12]:
# 递归加载目录下所有 .txt（glob 控制匹配）
docs = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.*",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
    use_multithreading=True,
).load()
show("DirectoryLoader", docs)

# 也支持懒加载（生成器，逐条产出，适合大数据集）
for doc in DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
).lazy_load():
    print("lazy_load ->", doc.metadata["source"])

100%|██████████| 5/5 [00:00<00:00, 2855.21it/s]

[DirectoryLoader] 5 个 Document
  page_content: '# 标题\n\n这是 Markdown 文档。\n\n- 要点一\n- 要点二\n'
  metadata: {'source': 'temp/source_loader_demo/notes.md'}
  page_content: '%PDF-1.4\n1 0 obj\n<< /Type /Catalog /Pages 2 0 R >>\nendobj\n2 0 obj\n<< /'
  metadata: {'source': 'temp/source_loader_demo/doc.pdf'}
  page_content: '[{"text": "Python 适合后端", "source": "a"}, {"text": "Vue 适合前端", "source"'
  metadata: {'source': 'temp/source_loader_demo/items.json'}
  page_content: 'name,role\n小明,后端\n小红,前端\n'
  metadata: {'source': 'temp/source_loader_demo/users.csv'}
  page_content: 'LangChain 是一个用于构建 LLM 应用的框架。\n它支持文档加载、切分、检索。'
  metadata: {'source': 'temp/source_loader_demo/hello.txt'}
lazy_load -> temp/source_loader_demo/hello.txt


#### 3. CSVLoader：逐行加载 CSV

In [4]:
# CSV：每一行变成一个 Document，page_content 是「列名: 值」，metadata 带 row
docs = CSVLoader(str(DATA_DIR / "users.csv"), encoding="utf-8").load()
show("CSVLoader", docs)

[CSVLoader] 2 个 Document
  page_content: 'name: 小明\nrole: 后端'
  metadata: {'source': 'temp/source_loader_demo/users.csv', 'row': 0}
  page_content: 'name: 小红\nrole: 前端'
  metadata: {'source': 'temp/source_loader_demo/users.csv', 'row': 1}


#### 4. JSONLoader：按 jq 表达式抽取 JSON 内容

In [5]:
# JSON：用 jq_schema 指定要抽取的内容（这里是每个元素的 text 字段）
docs = JSONLoader(str(DATA_DIR / "items.json"), jq_schema=".[].text").load()
show("JSONLoader", docs)

[JSONLoader] 2 个 Document
  page_content: 'Python 适合后端'
  metadata: {'source': '/Users/lijixu/PycharmProjects/langchain_demo/charpter11-RAG/temp/source_loader_demo/items.json', 'seq_num': 1}
  page_content: 'Vue 适合前端'
  metadata: {'source': '/Users/lijixu/PycharmProjects/langchain_demo/charpter11-RAG/temp/source_loader_demo/items.json', 'seq_num': 2}


#### 5. PyPDFLoader：按页加载 PDF

In [6]:
# PDF：按页加载，metadata 含 page / total_pages 等
docs = PyPDFLoader(str(DATA_DIR / "doc.pdf"), extraction_mode="layout").load()
print("页数:", len(docs))
print("内容:", docs[0].page_content)
print("metadata:", docs[0].metadata)

页数: 1
内容: Hello RAG PDF
metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'temp/source_loader_demo/doc.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


#### 6. WebBaseLoader：加载网页

In [7]:
# 网页：抓取并把 HTML 解析为纯文本（需要联网）
docs = WebBaseLoader("https://github.com/pydantic/pydantic-ai").load()
print("来源:", docs[0].metadata["source"])
print("标题:", docs[0].metadata.get("title"))
print("正文片段:", docs[0].page_content)

来源: https://github.com/pydantic/pydantic-ai
标题: GitHub - pydantic/pydantic-ai: How Python does AI. Agents, realtime voice, image generation, embeddings. Every model, every interface, typed end to end. · GitHub
正文片段: 




























































































































GitHub - pydantic/pydantic-ai: How Python does AI. Agents, realtime voice, image generation, embeddings. Every model, every interface, typed end to end. · GitHub


















































Skip to content

































Navigation MenuSign inAppearance settingsPlatformAI CODE CREATIONGitHub CopilotWrite better code with AIGitHub Copilot appDirect agents from issue to mergeMCP RegistryIntegrate external toolsDEVELOPER WORKFLOWSActionsAutomate any workflowCodespacesInstant dev environmentsIssuesPlan and track workCode ReviewManage code changesCode QualityEnforce quality at mergeAPPLICATION SECURITYGitHub Advanced

#### 7. Unstructured：结构化解析（Markdown / HTML / PDF…）

`Unstructured*Loader` 基于 `unstructured` 库，会把文档解析成**带类型的元素**
（Title / ListItem / NarrativeText…），并自动**去掉 Markdown 标记**，得到更干净的正文。

需要额外依赖（已加入本项目）：`uv add unstructured markdown`。

> 运行环境缺少 `libmagic` 时会提示 `libmagic is unavailable`，不影响文本解析；
> 如需更好的文件类型识别可执行 `brew install libmagic`。

In [11]:
# 解析 Markdown：去掉 #、- 等标记，保留正文结构
docs = UnstructuredMarkdownLoader(str(DATA_DIR / "notes.md"), mode="elements").load()
show("UnstructuredMarkdownLoader", docs)

# 通用加载器：自动识别文件类型
docs = UnstructuredFileLoader(str(DATA_DIR / "hello.txt")).load()
show("UnstructuredFileLoader", docs)

libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


[UnstructuredMarkdownLoader] 4 个 Document
  page_content: '标题'
  metadata: {'source': 'temp/source_loader_demo/notes.md', 'category_depth': 0, 'languages': ['zho'], 'file_directory': 'temp/source_loader_demo', 'filename': 'notes.md', 'filetype': 'text/markdown', 'last_modified': '2026-09-24T17:39:06', 'category': 'Title', 'element_id': 'c9fbe3a9a02715198f5827e5f74036d4'}
  page_content: '这是 Markdown 文档。'
  metadata: {'source': 'temp/source_loader_demo/notes.md', 'languages': ['pol', 'eng'], 'file_directory': 'temp/source_loader_demo', 'filename': 'notes.md', 'filetype': 'text/markdown', 'last_modified': '2026-09-24T17:39:06', 'parent_id': 'c9fbe3a9a02715198f5827e5f74036d4', 'category': 'UncategorizedText', 'element_id': 'd42edc906491cc26501ea64afffefbd8'}
  page_content: '要点一'
  metadata: {'source': 'temp/source_loader_demo/notes.md', 'category_depth': 1, 'languages': ['zho'], 'file_directory': 'temp/source_loader_demo', 'filename': 'notes.md', 'filetype': 'text/markdown', 'last_modifi

#### 补充：`yield` 与生成器（`lazy_load` 的原理）

`lazy_load` 里用到的 `yield` 会把函数变成**生成器（generator）**：函数可以分多次产出值，
每产出一个就「暂停」，下次从这里继续，而不是一次性 `return` 整个列表。

```python
def f_return():
    return [1, 2, 3]        # 一次造出整个列表

def f_yield():
    yield 1                # 产出 1 后暂停
    yield 2                # 被 next() 唤醒后产出 2
    yield 3
```

区别与作用：

- `return` 只返回一次并结束函数；`yield` 可以产出多次，每次暂停并保留现场；
- 调用生成器函数**不会立即执行**，而是返回一个生成器对象，由 `for` / `next()` 驱动它；
- **省内存、可惰性**：不必先把所有结果拼成一个大列表，需要一条才产一条。

对应到加载器：

```python
def lazy_load(self):
    yield Document(...)               # 每 next 一次产出一个 Document

docs = list(loader.lazy_load())       # 用 list() 把生成器的值收集起来
```

`BaseLoader.load()` 的实现其实就是 `return list(self.lazy_load())`（源码 `base.py:43`）——
所谓「一次性加载」只是把生成器**全部取出来装进列表**；而直接用 `lazy_load()`
可以边读边处理，适合大目录或大数据集。

#### 8. 自定义 Loader

内置加载器覆盖不到时，只要继承 `BaseLoader` 并实现 `lazy_load` 即可。（例如 `UnstructuredMarkdownLoader` 需要额外的 `unstructured` 依赖，这里用自定义 Loader 替代。）

In [9]:
# 自定义 Loader：实现 lazy_load 即可（BaseLoader 会据此提供 load / aload）
class MarkdownLoader(BaseLoader):
    """把 Markdown 文件整体加载为一个 Document。"""

    def __init__(self, file_path: str) -> None:
        self.file_path = file_path

    def lazy_load(self):
        text = Path(self.file_path).read_text(encoding="utf-8")
        yield Document(page_content=text, metadata={"source": self.file_path})


docs = list(MarkdownLoader(str(DATA_DIR / "notes.md")).lazy_load())
show("自定义 MarkdownLoader", docs)

[自定义 MarkdownLoader] 1 个 Document
  page_content: '# 标题\n\n这是 Markdown 文档。\n\n- 要点一\n- 要点二\n'
  metadata: {'source': 'temp/source_loader_demo/notes.md'}


#### 小结

| 加载器 | 来源 | 特点 |
| --- | --- | --- |
| `TextLoader` | 单个文本文件 | 最简单 |
| `DirectoryLoader` | 目录 | 用 `glob` 递归匹配，配合 `loader_cls` |
| `CSVLoader` | CSV | 每行一个 Document |
| `JSONLoader` | JSON | 用 `jq_schema` 抽取，需 `jq` |
| `PyPDFLoader` | PDF | 按页加载，metadata 含页码 |
| `WebBaseLoader` | 网页 | 抓取并解析 HTML |
| `Unstructured*Loader` | Markdown/多格式 | 结构化解析、去标记，需 `unstructured` |

**要点**

1. 一切加载结果都是 `Document(page_content, metadata)`；
2. `load()` 返回列表，`lazy_load()` 返回生成器；
3. 加载器的 `metadata` 会带上 `source`，是后续做引用溯源的依据；
4. 找不到现成加载器时，继承 `BaseLoader` 实现 `lazy_load` 即可。